PAra El Consumo

In [0]:
# Importaciones necesarias para trabajar en Databricks
import pandas as pd
# El objeto 'spark' ya está disponible globalmente en Databricks

# --- CONFIGURACIÓN DE RUTAS Y LISTA DE TABLAS ---
# AJUSTE ESTOS VALORES según su configuración de Catálogo y Esquema
CATALOGO = "workspace" 
ESQUEMA_GOLD = "gold"

# Lista de nombres de las tablas en la capa GOLD que se desean cargar
# Se asume que los nombres compuestos son tablas separadas o vistas
TABLAS_A_CARGAR = [
    "comparison_matrix",
    "logistics_analysis",
    "olist_orders_gold",
    "reporte_base",  # Asumido
    "reporte_final", # Asumido
    "resumen_negocio",
    "resumen_segmentos",
    "satisfaction_analysis_adv",
    "satisfaction_final",
    "segmentacion_clientes",
    "seller_analysis", # Asumido
    "top_5_geo",
    "top_categories",
    "top_productos"
]

# Diccionario para almacenar los DataFrames de Pandas (la clave será el nombre de la tabla)
gold_dataframes_pandas = {}

# --- ADVERTENCIA DE MEMORIA ---
# La función .toPandas() mueve todos los datos del cluster a un solo nodo.
# Si alguna tabla GOLD es demasiado grande, esto puede causar un error OOM (Out Of Memory).
print(f"ATENCIÓN: Se cargarán {len(TABLAS_A_CARGAR)} tablas desde la capa GOLD y se convertirán a Pandas.")
print("Asegúrese de que el tamaño total de los datos no exceda la memoria del nodo Driver.")
# 

# 1. Bucle de Carga y Conversión
for table_name in TABLAS_A_CARGAR:
    full_table_path = f"{CATALOGO}.{ESQUEMA_GOLD}.{table_name}"
    
    try:
        # Cargar la tabla de la capa GOLD en un Spark DataFrame
        spark_df_gold = spark.table(full_table_path)
        
        # 2. Contar filas para referencia
        row_count = spark_df_gold.count()

        # 3. Convertir el Spark DataFrame a un Pandas DataFrame
        df_pandas = spark_df_gold.toPandas()
        
        # 4. Almacenar el Pandas DataFrame en el diccionario
        gold_dataframes_pandas[table_name] = df_pandas
        
        print(f"[✅ ÉXITO] Tabla '{table_name}' ({row_count} filas) cargada como Pandas DataFrame.")

    except Exception as e:
        print(f"[❌ ERROR] No se pudo cargar la tabla '{table_name}'. Error: {e}")
        # La ejecución continúa con la siguiente tabla si falla una

print("\n--- RESUMEN DE CARGA ---")
print("DataFrames de Pandas cargados y disponibles en el diccionario 'gold_dataframes_pandas'.")
print("Para acceder, use: gold_dataframes_pandas['nombre_de_la_tabla']")

# --- NUEVO BUCLE DE PREVISUALIZACIÓN ---
print("\n--- PREVISUALIZACIÓN DE LOS PANDAS DATAFRAMES CARGADOS ---")

for table_name, df_pandas in gold_dataframes_pandas.items():
    # Imprimir un título claro antes de cada head()
    print(f"\n========================================================")
    print(f"PREVISUALIZACIÓN: {table_name}")
    print(f"========================================================\n")
    
    # Usar display() de Databricks para mostrar el head() de forma tabular y bonita
    # df_pandas.head() retorna un nuevo DataFrame, así que podemos usar display() directamente.
    display(df_pandas.head())